<a href="https://www.kaggle.com/code/abdallahahmed701/olist-ecommerce-end-to-end-analysis?scriptVersionId=319473131" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Chapter 1: The Great Merger & Spotting the Gaps 
**"We will use a Left Join to keep all orders, even those with missing details, to understand the integrity of our dataset before we start cleaning."**

In [ ]:
orders = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv')
items = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv')
customers = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv')
products = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv')
payments = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv')

In [ ]:
df = orders.merge(items, on='order_id', how='left') \
           .merge(customers, on='customer_id', how='left') \
           .merge(products, on='product_id', how='left') \
           .merge(payments, on='order_id', how='left') \
           .merge(reviews, on='order_id', how='left')

df

In [ ]:
print("--- Missing Values Report ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
print("\n--- Duplicates Report ---")
print(f"Total Duplicate Rows: {df.duplicated().sum()}")

In [ ]:
print("\n--- Data Types Check ---")
print(df[['order_purchase_timestamp', 'price']].dtypes)

In [ ]:
df.info()

# **Chapter 2: The Cleaning Operation**
**"Data cleaning is the most critical step in the pipeline. We found that our dates are stored as text, and several columns have missing values. In this chapter, we will fix these issues to ensure our future insights are accurate."**

In [ ]:
date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col])

df[date_columns].dtypes

**"Not all missing values are errors. For example, a missing delivery date might mean the order is still in transit or was canceled. We will fill missing review text with 'No Review' and remove only the rows that lack essential price information."**

In [ ]:
# 1. Fill missing review comments and titles
df['review_comment_title'] = df['review_comment_title'].fillna('No Title')
df['review_comment_message'] = df['review_comment_message'].fillna('No Message')

In [ ]:
# 2. Fill missing product categories with 'other'
df['product_category_name'] = df['product_category_name'].fillna('other')

In [ ]:
# 3. Drop rows where we have NO product_id or price (833 rows)
# Because we cannot analyze a sale without a price!
df.dropna(subset=['product_id', 'price'], inplace=True)

In [ ]:
print("Missing values handled!")
print(f"Current rows in dataset: {len(df)}")

**"Even after initial cleaning, some missing values remain. However, these are Logical Gaps. For example, missing delivery dates (order_delivered_customer_date) occur because some orders are still 'shipped' or 'invoiced' but not yet 'delivered'. Similarly, missing reviews mean the customer chose not to leave feedback. We will treat these carefully to maintain data integrity."**

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset=['payment_type'], inplace=True)

In [ ]:
cols_to_fix = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in cols_to_fix:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df['product_name_lenght'] = df['product_name_lenght'].fillna(0)
df['product_description_lenght'] = df['product_description_lenght'].fillna(0)
df['product_photos_qty'] = df['product_photos_qty'].fillna(0)

In [ ]:
print("Remaining logical gaps handled!")
print(f"Final Row Count for Chapter 2: {len(df)}")

# Outlier Detection & Business Anomalies
**"Now that our data is clean and structured, we need to ensure its statistical reliability. In this chapter, we will use the Interquartile Range (IQR) method to detect price outliers. We will also investigate 'Business Anomalies', such as orders where the shipping cost (Freight) exceeds the product price."**

In [ ]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

price_outliers = df[df['price'] > upper_bound]

print(f"Statistical Summary for Price:")
print(f"Upper Bound (Max logical price): ${upper_bound:.2f}")
print(f"Number of Outliers: {len(price_outliers)} out of {len(df)}")
print(f"Percentage of Outliers: {(len(price_outliers)/len(df))*100:.2f}%")

**"Beyond statistics, we must look at Business Logic. We will flag orders where the freight value is higher than the product price, as these cases might indicate logistics issues or data entry errors."**

In [ ]:
high_freight_orders = df[df['freight_value'] > df['price']]

print(f"--- Business Anomaly Report ---")
print(f"Orders with Shipping > Price: {len(high_freight_orders)}")
print(f"Max Freight Value found: ${df['freight_value'].max():.2f}")

# Chapter 4: Feature Engineering & Time-Series Preparation
**"To unlock deeper insights, we need to transform raw timestamps into meaningful metrics. In this chapter, we will calculate Delivery Performance (the difference between estimated and actual delivery) and extract time features (Hour, Day, Month) to understand shopping patterns in Brazil."**

In [ ]:
df['delivery_time_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

In [ ]:
df['delivery_diff_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days

In [ ]:
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_day_name'] = df['order_purchase_timestamp'].dt.day_name()
df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour

In [ ]:
print("Chapter 4: Time-based features created successfully!")
print(df[['delivery_time_days', 'delivery_diff_days', 'purchase_day_name']].head())

# Chapter 5: Visualizing Business Insights (EDA)
"Visualization is where data tells its story. In this chapter, we explore three main pillars:

1.Seasonality: When do people shop the most?

2.Logistics: How reliable is our delivery system?

3.Product Performance: Which categories drive our revenue?"

In [ ]:
hourly_orders = df.groupby('purchase_hour').size().reset_index(name='order_count')

fig1 = px.area(hourly_orders, x='purchase_hour', y='order_count', 
               title='Peak Shopping Hours in Brazil ',
               labels={'purchase_hour': 'Hour of Day (24h)', 'order_count': 'Total Orders'},
               color_discrete_sequence=['teal'])

fig1.update_layout(hovermode="x unified")
fig1.show()

Business Insight: The Afternoon Rush

Trend: Shopping activity starts to surge at 10:00 AM and remains consistently high until 10:00 PM.

Peak: The absolute peak occurs around 4:00 PM (16h).

Actionable Strategy: Marketing campaigns and "Flash Sales" should be scheduled between 10 AM and 4 PM to maximize reach and conversion rates

In [ ]:
fig2 = px.histogram(df, x="delivery_diff_days", 
                   title="Delivery Performance: Early vs Late ",
                   labels={'delivery_diff_days': 'Days Difference (Negative = Early)'},
                   color_discrete_sequence=['purple'],
                   nbins=100,
                   marginal="box") 

fig2.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Deadline")

fig2.show()

Business Insight: Logistics Efficiency & Trust

Reliability: The vast majority of the distribution lies to the left of the red deadline line, meaning most customers receive their orders much earlier than expected.

Outlier Alert: The Box Plot above the histogram reveals several late delivery outliers (the dots on the right). These specific cases should be investigated for potential carrier issues or regional logistics bottlenecks.

Metric: This early delivery trend is a key driver for customer satisfaction and high review scores.

In [ ]:
top_cats = df.groupby('product_category_name')['price'].sum().sort_values(ascending=False).head(10).reset_index()

fig3 = px.bar(top_cats, x='price', y='product_category_name', 
             orientation='h', 
             title='Top 10 Categories by Revenue',
             labels={'price': 'Total Revenue ($)', 'product_category_name': 'Category'},
             color='price', 
             color_continuous_scale='Viridis')

fig3.show()

Business Insight: Revenue Drivers

Dominance: The 'beleza_saude' (Health & Beauty) and 'relogios_presentes' (Watches & Gifts) categories are the top revenue generators, surpassing 1.2M each.

Diversification: Olist has a healthy mix of lifestyle (Beauty), home (Bed/Bath), and tech (Computers/Accessories) products.

Strategy: Cross-selling beauty products with watches/gifts could be a high-potential strategy given their shared popularity.

In [ ]:
state_orders = df.groupby('customer_state').size().reset_index(name='order_count')

fig_map = px.choropleth(state_orders, 
                        locations='customer_state', 
                        locationmode="USA-states", 
                        color='order_count',
                        scope="south america", 
                        title='Orders Distribution by Brazilian State',
                        labels={'order_count': 'Total Orders', 'customer_state': 'State'},
                        color_continuous_scale='Viridis')

state_orders = state_orders.sort_values('order_count', ascending=False)
fig_state = px.bar(state_orders, x='customer_state', y='order_count',
                   color='order_count',
                   title='Total Orders per State',
                   labels={'order_count': 'Number of Orders', 'customer_state': 'State Code'},
                   color_continuous_scale='Turbo')

fig_state.show()

Business Insight: Geographic Concentration & Market Dominance

The Power of São Paulo: The state of SP (São Paulo) accounts for the vast majority of orders, nearly triple the volume of the next state, MG (Minas Gerais).

Strategic Logistics: This high concentration suggests that the main distribution center should be located in or near São Paulo to minimize shipping costs and delivery times for the bulk of customers.

Expansion Opportunity: While SP is the primary market, there is a significant long-tail of states with lower order volumes, indicating a potential opportunity for targeted regional marketing to increase national market share.